# The Kernel Trick

This notebook verifies the central kernel idea from Lecture 6: an inner product in an expanded feature space can sometimes be computed directly from the original inputs.

## 1. Explicit polynomial feature map

For two-dimensional inputs, define the feature map corresponding exactly to the degree-two polynomial kernel $(1+x^Tz)^2$:

$$\phi(x_1,x_2) = (1,\;\sqrt{2}x_1,\;\sqrt{2}x_2,\;x_1^2,\;x_2^2,\;\sqrt{2}x_1x_2).$$

Then $\phi(x)^T\phi(z)=(1+x^Tz)^2$.

In [1]:
import numpy as np

def phi(x):
    x1, x2 = x
    return np.array([1.0, np.sqrt(2) * x1, np.sqrt(2) * x2, x1**2, x2**2, np.sqrt(2) * x1 * x2])

x = np.array([1.5, -0.5])
z = np.array([-0.25, 2.0])

explicit_inner_product = phi(x) @ phi(z)
dot = x @ z
kernel_value = (1 + dot)**2

print('phi(x) =', phi(x))
print('phi(z) =', phi(z))
print('Explicit feature-space inner product:', explicit_inner_product)
print('Polynomial kernel:', kernel_value)
print('Difference:', abs(explicit_inner_product - kernel_value))
assert np.allclose(explicit_inner_product, kernel_value)

phi(x) = [ 1.          2.12132034 -0.70710678  2.25        0.25       -1.06066017]
phi(z) = [ 1.         -0.35355339  2.82842712  0.0625      4.         -0.70710678]
Explicit feature-space inner product: 0.14062499999999967
Polynomial kernel: 0.140625
Difference: 3.3306690738754696e-16


The two values agree up to floating-point precision. The important point is not the particular polynomial: it is that the high-dimensional inner product can be evaluated without explicitly constructing the expanded vectors.

## 2. Compare explicit features with the kernel

A kernel is defined by

$$K(x,z)=\phi(x)^T\phi(z).$$

For the map above, the corresponding kernel is

$$K(x,z)=(1+x^Tz)^2.$$

The kernel computation needs only the original dot product and a small amount of scalar arithmetic.

In [2]:
def polynomial_kernel(X, Z, degree=2):
    return (1 + X @ Z) ** degree

X = np.array([[1, 2], [2, 1], [-1, -1], [0.5, 1.5]])
Z = np.array([[0.5, -1], [1.0, 1.0]])

K = polynomial_kernel(X, Z, degree=2)
print(K)

[[12.25    4.    ]
 [ 9.      0.    ]
 [ 0.25    1.    ]
 [ 7.5625  4.    ]]


## 3. A higher-order kernel

The same idea can be extended. Polynomial kernels of the form

$$K(x,z)=(1+x^Tz)^p$$

correspond to increasingly rich polynomial feature representations. The feature dimension can grow very quickly, while evaluating the kernel still requires only the original dot product followed by a power.

In [3]:
degrees = [1, 2, 3, 5, 10]
dot = x @ z

for p in degrees:
    print(f'degree={p:2d}: K(x,z)={(1 + dot)**p:.6f}')

degree= 1: K(x,z)=-0.375000
degree= 2: K(x,z)=0.140625
degree= 3: K(x,z)=-0.052734
degree= 5: K(x,z)=-0.007416
degree=10: K(x,z)=0.000055


## 4. Choosing the polynomial degree

The degree $p$ is not a value that the kernel trick determines automatically. It is a **hyperparameter**: we choose a set of candidate values and use validation or cross-validation to decide which one generalizes best.

The reason is that $p$ controls the complexity of the implicit feature representation. Larger degrees allow richer polynomial interactions, but greater flexibility does not guarantee better performance on unseen data.

A typical model-selection workflow is:

1. choose candidate degrees, such as $p \\in \\{1,2,3,4,5\\}$;
2. train the classifier using each degree;
3. evaluate each model on validation data or with cross-validation;
4. choose the degree with the best validation performance;
5. retrain the selected model on the available training data before the final test evaluation.

In notation, if $S_{\\mathrm{val}}(p)$ is the validation score for degree $p$, then we choose

$$
p^* = \\arg\\max_p S_{\\mathrm{val}}(p)
$$

The kernel trick makes evaluating different degrees inexpensive with respect to feature construction: for each candidate $p$, we can compute $(1+x^Tz)^p$ directly instead of explicitly constructing the corresponding polynomial feature vectors.

This connects the kernel trick to the model-selection idea introduced earlier in the course: **the learning algorithm fits the model, while validation selects the hyperparameters that control the model.**

The examples above use degrees $1,2,3,5,10$ only to illustrate how the kernel value changes. Those values are not a claim that one degree is universally best; the appropriate degree depends on the dataset and should be selected using validation.



### Takeaway

The kernel trick is a computational shortcut:

$$\phi(x)^T\phi(z) \;\longrightarrow\; K(x,z)$$

When an algorithm can be written using only feature-space inner products, it can often operate implicitly in the expanded space.